In [1]:
# --- 1. (ใหม่!) Imports ---
# import torch # <<< ไม่ต้องใช้
# from transformers import pipeline # <<< ไม่ต้องใช้
import json
import textwrap
import os
from openai import OpenAI # <<< เพิ่ม OpenAI
import re # <<< (ใหม่!) Import Regular Expressions สำหรับการ sorting

print("🚀 Initializing the Multi-Turn LLM Therapist (BASELINE - GPT-4o Mini)...")
print("="*50)

# --- 2. (ใหม่!) Setup OpenAI Client ---
print("Setting up OpenAI client...")

# <<< ⭐️⭐️⭐️ ไดจิต้องใส่ Key ตรงนี้ ⭐️⭐️⭐️ >>>
# ใส่ Key จริงๆ ของไดจิแทน 'sk-xxxx...' นะคะ
os.environ['OPENAI_API_KEY'] = "PUT_YOUR_KEY_HERE" 

if "OPENAI_API_KEY" not in os.environ:
    print("="*50)
    print("Error: OPENAI_API_KEY not found...")
    print("="*50)
    
client = OpenAI()
OPENAI_MODEL_ID = "gpt-4o-mini"
print(f"✅ OpenAI client initialized for model: {OPENAI_MODEL_ID}")

# --- 3. โหลดข้อมูลจาก "สุดยอดคัมภีร์" (เหมือนเดิม) ---
json_path = '../merged_prompts_to_LLMs_therapist/final_llm_inputs.json'
try:
    with open(json_path, 'r', encoding='utf-8') as f:
        merged_data = json.load(f)
    print(f"✅ Successfully loaded {len(merged_data)} user data entries.")
except FileNotFoundError:
    print(f"❌ ERROR: Cannot find '{json_path}'.")

# --- 4. (ใหม่!) แยก Prompt Template สำหรับ OpenAI (ฉบับ Baseline) ---
# ** (แก้ไข) **: เอา [Vocal Context] ออกจาก Prompt
SYSTEM_PROMPT_BASELINE = """You are "Luna AI Therapist," an empathetic, warm, and observant AI assistant. You are non-judgmental, and your primary goal is to respond with compassion. You must utilize the provided 'Transcript' of what the user said to understand their emotional state.

[Your Task]
Your task is to directly craft the response for Luna AI Therapist. DO NOT generate any code, explanations, or surrounding text. Your entire output must ONLY be the therapeutic response itself, starting immediately with the words of the therapist. Your response must be brief (2-3 sentences), warm, and end with a single, gentle, open-ended question.
"""

USER_PROMPT_TEMPLATE_BASELINE = """Here is the data from a user session for you to analyze:

[Transcript]
"{transcript}"
"""
print("✅ Baseline prompt templates are ready (Text-Only).")
print("="*50)


# --- 5. (ใหม่!) สร้างลำดับการสนทนา (Dynamic Sorting) ---
# (เหมือนกับที่เราทำในไฟล์ Notebook หลักเลยค่ะ)

data_map = {item['file_name']: item for item in merged_data}

def get_sort_keys(filename):
    """
    Extracts (dialogue_num, utterance_num) as integers for correct sorting.
    """
    match = re.search(r'dialogue_(\d+)_utterance_(\d+)\.wav', filename)
    if match:
        dialogue_num = int(match.group(1))
        utterance_num = int(match.group(2))
        return (dialogue_num, utterance_num)
    else:
        return (999, 999)

all_filenames = list(data_map.keys())
dialogue_sequence = sorted(all_filenames, key=get_sort_keys) # <<< เรียงลำดับทั้งหมด!

print(f"✅ Found and sorted {len(dialogue_sequence)} utterances from the data map.")
print("="*50)

# <<< (ใหม่!) สร้าง List ว่างๆ เพื่อรอเก็บผลลัพธ์ Baseline >>>
all_baseline_results = []
print("Starting BASELINE simulation... Results will be collected.")


# --- 6. (อัปเกรด) เริ่มการจำลองบทสนทนา (ใช้ OpenAI) ---
for i, filename in enumerate(dialogue_sequence):
    print(f"\n\n--- TURN {i+1}/{len(dialogue_sequence)} ---")
    
    turn_data = data_map.get(filename)
    if not turn_data:
        print(f"⚠️  Warning: Data for {filename} not found. Skipping turn.")
        continue

    # ** (แก้ไข) **: เอา print() ของ Vocal Context ออก
    print(f"CLIENT SAID: \"{turn_data['transcript']}\"")
    
    # (ใหม่!) สร้าง User Prompt สำหรับเทิร์นนี้ (แบบ Baseline)
    user_prompt = USER_PROMPT_TEMPLATE_BASELINE.format(
        transcript=turn_data['transcript']
    )
    
    # (ใหม่!) สร้าง list ของ messages ที่จะส่ง
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_BASELINE},
        {"role": "user", "content": user_prompt}
    ]
    
    response_text = ""
    try:
        # (ใหม่!) สั่งให้ AI ตอบกลับ (ใช้ OpenAI)
        completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        response_text = completion.choices[0].message.content.strip()
        
    except Exception as e:
        response_text = f"ERROR: Failed to get response from OpenAI. {e}"
    
    # # (เดิม) แสดงผลลัพธ์
    # print("\nTHERAPIST'S RESPONSE (BASELINE):")
    # wrapped_text = textwrap.fill(response_text, width=80)
    # print(wrapped_text)
    # print("="*50)
    
    # <<< (ใหม่!) เก็บผลลัพธ์ของเทิร์นนี้ลงใน List >>>
    result_entry = {
        "file_name": filename,
        "transcript": turn_data['transcript'],
        # "emotion_caption": "N/A (Baseline)", # ไม่มีใน Baseline
        "therapist_response_baseline": response_text 
    }
    all_baseline_results.append(result_entry)
    # <<< สิ้นสุดส่วนที่เพิ่ม >>>

print("\n🎉 Baseline multi-turn simulation complete!")
print(f"Collected {len(all_baseline_results)} baseline results. Ready to be saved in the next cell.")

🚀 Initializing the Multi-Turn LLM Therapist (BASELINE - GPT-4o Mini)...
Setting up OpenAI client...
✅ OpenAI client initialized for model: gpt-4o-mini
✅ Successfully loaded 300 user data entries.
✅ Baseline prompt templates are ready (Text-Only).
✅ Found and sorted 300 utterances from the data map.
Starting BASELINE simulation... Results will be collected.


--- TURN 1/300 ---
CLIENT SAID: "Um, I've been really anxious about going back to the animal shelter where I volunteer. I feel like the animals will hate me because they didn't remember me the last time I visited. It's been really tough."


--- TURN 2/300 ---
CLIENT SAID: "Well, I went to the shelter a few months ago, and some of the animals that used to greet me warmly didn't recognize me. It felt like a punch to the gut. Since then, I've been avoiding going back because I can't handle the thought of being rejected by them."


--- TURN 3/300 ---
CLIENT SAID: "and affecting my sleep i keep replaying that visit in my mind my passion

In [2]:
# Save output

import json

# ตั้งชื่อไฟล์ที่เราจะเซฟ (เติม _BASELINE)
output_save_path = "therapist_simulation_results_BASELINE.jsonl"

try:
    # ตรวจสอบว่าตัวแปร all_baseline_results (จาก Cell 1) มีอยู่จริงไหม
    if 'all_baseline_results' in locals() and all_baseline_results:
        print(f"Found {len(all_baseline_results)} results. Saving to '{output_save_path}'...")
        
        # วนลูปบันทึกผลลัพธ์ทีละบรรทัด (แบบ .jsonl)
        with open(output_save_path, 'w', encoding='utf-8') as f:
            for entry in all_baseline_results:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')
                
        print(f"✅ Successfully saved baseline results to '{output_save_path}'")
    else:
        print("⚠️ Warning: Could not find 'all_baseline_results' variable or it is empty.")
        print("Please make sure you have run the baseline simulation cell (Cell 1) successfully first.")

except Exception as e:
    print(f"❌ Error saving file: {e}")

Found 300 results. Saving to 'therapist_simulation_results_BASELINE.jsonl'...
✅ Successfully saved baseline results to 'therapist_simulation_results_BASELINE.jsonl'
